# SustentIA — Asistente Inteligente con Llama

**Hackathon 1 — Curso «Inteligencia Artificial Aplicada con Llama» (INFOTEC / DEVF — Módulo 1)**

---

## Contexto del proyecto

- **Nombre:** SustentIA — Asistente Inteligente con Llama
- **Problema real:** Los prospectos de la inmobiliaria **Sustentia** (Morelia, Michoacán) suelen escribir fuera de horario laboral buscando información inmediata sobre casas, departamentos o lotes. Cuando nadie responde a tiempo ni filtra su necesidad, los asesores pierden prospectos valiosos.
- **Usuario claro:** El prospecto que busca comprar una casa, departamento o lote residencial en Morelia o su zona conurbada.
- **Solución:** Un asistente conversacional impulsado por **Llama (`meta-llama/llama-3.1-8b-instruct`)** de Meta vía **OpenRouter** como proveedor principal, con **respaldo automático en Groq (`openai/gpt-oss-120b`)** y un **mensaje predefinido de espera con escalamiento a un asesor** si ningún modelo está disponible, que atiende de forma inmediata:
  1. Saluda cordialmente al prospecto.
  2. Identifica su necesidad (tipo de propiedad, zona y presupuesto).
  3. Recomienda opciones reales del catálogo de Sustentia sin inventar datos.
  4. Responde preguntas frecuentes (por ejemplo, créditos aceptados).
  5. Recuerda el nombre y presupuesto durante la conversación.
  6. Captura sus datos de contacto (nombre y teléfono) y propone agendar una visita.
  7. Escala a un asesor humano cuando el prospecto lo solicita o plantea dudas legales/fiscales.

> **Transparencia sobre los proveedores y modelos:**
> - **Principal:** **OpenRouter** (`https://openrouter.ai/api/v1`) con **`meta-llama/llama-3.1-8b-instruct`** (o `meta-llama/llama-3.3-70b-instruct`), usado como puerta de acceso a los modelos Llama de Meta porque Groq retiró el acceso *self-serve* a sus modelos Llama de chat.
> - **Respaldo automático de continuidad:** **Groq** (`https://api.groq.com/openai/v1`) con **`openai/gpt-oss-120b`** usando el mismo SDK compatible con OpenAI. Con total honestidad: **`openai/gpt-oss-120b` NO es un modelo Llama**; su función es mantener la continuidad operativa si OpenRouter falla o agota sus créditos.
> - **Sin modelo disponible (mensaje de espera):** Si OpenRouter deja de responder y el respaldo de Groq también falla, el prospecto recibe un mensaje predefinido —«En este momento no podemos atenderte, pero a primera hora un asesor se comunicará contigo»— y la conversación queda marcada para que un asesor la atienda.

## 1. Instalación de dependencias y conexión (OpenRouter principal + Groq respaldo)

En esta sección instalamos el SDK oficial de `openai` (si aún no está disponible y no estamos en modo sin instalación) y configuramos la cadena de proveedores **`OpenRouter (Llama) → Groq (respaldo) → mensaje de espera`** sobre la interfaz compatible con OpenAI:

1. **Proveedor principal — OpenRouter (`https://openrouter.ai/api/v1`):**
   - Modelo por defecto: **`meta-llama/llama-3.1-8b-instruct`** (el Llama 3.1 8B del curso).
   - Alternativa documentada: **`meta-llama/llama-3.3-70b-instruct`**, que sigue las reglas del prompt con mayor fidelidad (verificado en pruebas reales: el 8B llegó a inventar el nombre de un asesor y a sugerir un desarrollo fuera de presupuesto; el 70B no).
   - Se utiliza OpenRouter para Llama porque Groq retiró el acceso *self-serve* a sus modelos Llama de chat.
2. **Respaldo automático — Groq (`https://api.groq.com/openai/v1`):**
   - Modelo de respaldo: **`openai/gpt-oss-120b`**.
   - **Nota honesta de diseño:** `openai/gpt-oss-120b` **NO es un modelo Llama**; es el modelo disponible en la cuenta de Groq que actúa como red de seguridad para mantener la continuidad operativa si OpenRouter devuelve error (`402` sin créditos, `401`, `429`, error de red o servidor).
3. **Sin modelo disponible — Mensaje predefinido de espera:**
   - Si OpenRouter deja de responder y el respaldo de Groq también falla, el prospecto recibe un mensaje predefinido —«En este momento no podemos atenderte, pero a primera hora un asesor se comunicará contigo»— y la conversación queda marcada (`escalado=True`) para que un asesor la atienda.

**Seguridad de las API keys y operación parcial:**
- En Google Colab, `OPENROUTER_API_KEY` y `GROQ_API_KEY` se leen de forma segura desde **Secretos** (`google.colab.userdata`), desde variables de entorno o mediante `getpass`.
- **Nunca** se escriben llaves dentro del código fuente.
- Si solo hay una llave configurada (`OPENROUTER_API_KEY` o `GROQ_API_KEY`), el asistente trabaja con ese proveedor y el otro escalón se salta automáticamente.
- Si no hay llaves configuradas, la celda de demostración avisa que se necesita configurar `OPENROUTER_API_KEY` o `GROQ_API_KEY` y termina sin fingir una conversación; en una conversación real (llamada directa a `conversar()`), el prospecto recibe `MENSAJE_SIN_MODELO` y la conversación queda marcada como escalada para que un asesor la atienda.

In [ ]:
# Dependencias estándar de Python y utilidad para instalación compatible con Colab
import getpass
import importlib.util
import os
import re
import subprocess
import sys
import unicodedata

# Instalación silenciosa de `openai` en Python puro (evita magics `!pip` para mantener
# compatibilidad total con `py_compile` y entornos sin acceso a red).
if os.environ.get("SUSTENTIA_SIN_INSTALAR") != "1" and importlib.util.find_spec("openai") is None:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "openai"])
    except Exception as exc:
        print(f"Aviso: no se pudo instalar 'openai' automáticamente ({exc}). Sin modelo disponible, se enviará el mensaje de espera y se marcará la conversación como escalada.")

try:
    from openai import OpenAI
except ImportError:
    OpenAI = None

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
MODELO_LLAMA = "meta-llama/llama-3.1-8b-instruct"
# Alternativa con mayor fidelidad a las reglas del prompt (verificado en pruebas):
# MODELO_LLAMA = "meta-llama/llama-3.3-70b-instruct"

GROQ_BASE_URL = "https://api.groq.com/openai/v1"
MODELO_RESPALDO_GROQ = "openai/gpt-oss-120b"


def _obtener_api_key_secreto(
    nombre_secreto: str,
    nombre_proveedor: str,
    permitir_prompt: bool = True,
) -> str:
    """Obtiene una API key desde Colab userdata, variable de entorno o getpass."""
    try:
        from google.colab import userdata  # type: ignore

        clave_colab = userdata.get(nombre_secreto)
        if clave_colab and clave_colab.strip():
            return clave_colab.strip()
    except Exception:
        pass

    clave_entorno = os.environ.get(nombre_secreto, "").strip()
    if clave_entorno:
        return clave_entorno

    if (
        permitir_prompt
        and ("google.colab" in sys.modules or sys.stdin.isatty())
    ):
        try:
            clave_manual = getpass.getpass(
                f"Ingresa tu API key de {nombre_proveedor} ({nombre_secreto}, o presiona Enter para omitir): "
            )
            return clave_manual.strip()
        except (EOFError, KeyboardInterrupt, Exception):
            return ""

    return ""


def obtener_api_key(permitir_prompt: bool = True) -> str:
    """Obtiene la API key principal de OpenRouter (`OPENROUTER_API_KEY`)."""
    return _obtener_api_key_secreto(
        "OPENROUTER_API_KEY",
        "OpenRouter",
        permitir_prompt=permitir_prompt,
    )


def obtener_api_key_groq(permitir_prompt: bool = True) -> str:
    """Obtiene la API key de respaldo de Groq (`GROQ_API_KEY`)."""
    return _obtener_api_key_secreto(
        "GROQ_API_KEY",
        "Groq (respaldo)",
        permitir_prompt=permitir_prompt,
    )


def crear_cliente_openrouter(api_key: str | None = None):
    """Crea el cliente principal de OpenRouter vía SDK `openai` si existen paquete y clave."""
    if OpenAI is None:
        return None
    clave = api_key if api_key is not None else obtener_api_key(permitir_prompt=True)
    if not clave:
        return None
    return OpenAI(base_url=OPENROUTER_BASE_URL, api_key=clave)


def crear_cliente_groq(api_key: str | None = None):
    """Crea el cliente de respaldo en Groq vía SDK `openai` si existen paquete y clave."""
    if OpenAI is None:
        return None
    clave = api_key if api_key is not None else obtener_api_key_groq(permitir_prompt=True)
    if not clave:
        return None
    return OpenAI(base_url=GROQ_BASE_URL, api_key=clave)


def describir_proveedor_activo(
    cliente_or,
    cliente_gr,
) -> str:
    """Devuelve una descripción clara de la disponibilidad de OpenRouter y Groq."""
    if cliente_or is None and cliente_gr is None:
        return "Sin proveedores disponibles (OpenRouter y Groq sin llaves configuradas)"
    if cliente_or is not None and cliente_gr is not None:
        return (
            f"OpenRouter ({MODELO_LLAMA}) + respaldo automático Groq ({MODELO_RESPALDO_GROQ})"
        )
    if cliente_or is not None:
        return f"OpenRouter ({MODELO_LLAMA}) [sin respaldo Groq]"
    return f"Groq ({MODELO_RESPALDO_GROQ}) [OpenRouter no configurado]"


cliente_openrouter = crear_cliente_openrouter()
cliente_groq = crear_cliente_groq()
PROVEEDOR_ACTIVO = describir_proveedor_activo(cliente_openrouter, cliente_groq)
MODO_EJECUCION = PROVEEDOR_ACTIVO

print(f"Principal: OpenRouter ({OPENROUTER_BASE_URL}) | Modelo Llama: {MODELO_LLAMA} | Listo: {cliente_openrouter is not None}")
print(f"Respaldo: Groq ({GROQ_BASE_URL}) | Modelo continuidad (no Llama): {MODELO_RESPALDO_GROQ} | Listo: {cliente_groq is not None}")
print("Sin modelo disponible: mensaje predefinido de espera + escalamiento a asesor")
print(f"Proveedor activo: {PROVEEDOR_ACTIVO}")

## 2. Catálogo oficial de desarrollos de Sustentia

Definimos los **7 desarrollos reales** como una lista de diccionarios estructurados (fuente única de verdad).

**Reglas de negocio del catálogo:**
- Créditos aceptados en general: **Infonavit**, **Fovissste** y **bancario**.
- **No inventar** unidades disponibles ni cantidades exactas.
- **No dar direcciones exactas**, únicamente las zonas indicadas en el catálogo.

In [ ]:
# Constantes de negocio y contacto (sin teléfonos personales ni direcciones exactas)
CREDITOS_ACEPTADOS = ["Infonavit", "Fovissste", "Bancario"]
CONTACTO_ASESOR = "[teléfono del asesor]"
MENSAJE_ESCALAMIENTO = "Te pongo en contacto con un asesor, un momento, por favor."
MENSAJE_SIN_MODELO = "En este momento no podemos atenderte, pero a primera hora un asesor se comunicará contigo."

# Catálogo oficial de desarrollos de Sustentia
CATALOGO_DESARROLLOS = [
    {
        "nombre": "San Pedro",
        "tipo": "casas",
        "zona": "Morelia, zona Walmart La Huerta",
        "precio_desde": 1250000,
        "precio_texto": "desde $1,250,000",
        "recamaras": "2 a 3 recámaras",
        "detalles": "2 a 3 recámaras; entrega inmediata en algunos modelos",
        "creditos": CREDITOS_ACEPTADOS,
    },
    {
        "nombre": "Torres Zojo",
        "tipo": "departamentos",
        "zona": "Morelia, por el Tecnológico de Morelia",
        "precio_desde": 1200000,
        "precio_texto": "desde $1,200,000 aprox.",
        "recamaras": None,
        "detalles": "Departamentos ubicados por el Tecnológico de Morelia",
        "creditos": CREDITOS_ACEPTADOS,
    },
    {
        "nombre": "Veronesa",
        "tipo": "casas",
        "zona": "Morelia, zona La Calera",
        "precio_desde": 2690000,
        "precio_texto": "desde $2,690,000",
        "recamaras": "3 recámaras",
        "detalles": "3 recámaras",
        "creditos": CREDITOS_ACEPTADOS,
    },
    {
        "nombre": "Paseo de Andalucía (modelos Alba y Leira)",
        "tipo": "departamentos y casas",
        "zona": "Morelia, zona Altozano",
        "precio_desde": 2830000,
        "precio_texto": "desde $2,830,000",
        "recamaras": None,
        "detalles": "Modelos Alba y Leira (departamentos y casas)",
        "creditos": CREDITOS_ACEPTADOS,
    },
    {
        "nombre": "Rincón de las Aves",
        "tipo": "casas",
        "zona": "Morelia, zona El Vergel",
        "precio_desde": 2245000,
        "precio_texto": "desde $2,245,000",
        "recamaras": None,
        "detalles": "Casa club y gimnasio",
        "creditos": CREDITOS_ACEPTADOS,
    },
    {
        "nombre": "Tepatzícuaro",
        "tipo": "lotes residenciales campestres",
        "zona": "Acuitzio del Canje, a 20 min de Morelia",
        "precio_desde": 705127,
        "precio_texto": "desde $705,127",
        "recamaras": None,
        "detalles": "Lago, trotapista y ciclovía",
        "creditos": CREDITOS_ACEPTADOS,
    },
    {
        "nombre": "Valle Verde",
        "tipo": "lotes",
        "zona": "Morelia",
        "precio_desde": None,
        "precio_por_m2": 5000,
        "precio_texto": "$5,000 por m²",
        "recamaras": None,
        "detalles": "Lotes en Morelia con precio por metro cuadrado",
        "creditos": CREDITOS_ACEPTADOS,
    },
]

print(f"Desarrollos cargados en catálogo: {len(CATALOGO_DESARROLLOS)}")
for d in CATALOGO_DESARROLLOS:
    print(f" - {d['nombre']} ({d['tipo']}) | {d['zona']} | {d['precio_texto']}")

## 3. Estrategia de prompts (y por qué de cada decisión)

En lugar de escribir un prompt genérico, diseñamos el **`SYSTEM_PROMPT` como un contrato operativo** para Llama (`meta-llama/llama-3.1-8b-instruct`) vía OpenRouter. Cada bloque responde a un riesgo específico del dominio inmobiliario:

| Componente del prompt | Decisión de diseño | ¿Por qué se eligió así? |
| :--- | :--- | :--- |
| **Rol y tono** | Asistente virtual de Sustentia en Morelia, amable, claro y conciso. | Genera confianza inmediata fuera de horario y evita respuestas robóticas o demasiado largas para chat. |
| **Catálogo inyectado como contexto** | Serialización automática desde `CATALOGO_DESARROLLOS` hacia el `system prompt`. | Garantiza una **única fuente de verdad** (grounding): si cambia un precio o desarrollo en el diccionario, el prompt se actualiza sin duplicar información. |
| **Reglas negativas anti-alucinación** | Prohibición explícita de inventar unidades disponibles, cantidades exactas o direcciones exactas; uso obligatorio de `[teléfono del asesor]`. | Los LLM tienden a completar huecos cuando un cliente pregunta «¿cuántas casas quedan?» o «¿cuál es la calle?». Las restricciones explícitas frenan alucinaciones y protegen la privacidad. |
| **Flujo guiado de captura de datos** | Orden lógico: saludar → identificar tipo, zona y presupuesto → recomendar → resolver dudas de crédito → pedir nombre y teléfono → proponer visita. | Evita interrogar al prospecto con formularios fríos desde el primer mensaje; primero aporta valor recomendando y luego solicita el contacto. |
| **Regla de escalamiento + defensa en profundidad** | Instrucción textual en el prompt respaldada por `detectar_escalamiento()` en Python antes de llamar al modelo. | Temas legales/fiscales (ISR, escrituras, notaría) o peticiones de hablar con un humano requieren exactitud determinista en la frase de salida: *«Te pongo en contacto con un asesor, un momento, por favor.»* |
| **Temperatura baja (`0.3`)** | Parámetro conservador al invocar la API compatible con OpenAI. | Mantiene apego estricto a los precios, zonas y nombres reales del catálogo en lugar de improvisar variantes. |

**Elección de modelo y fidelidad al prompt:** Aunque `meta-llama/llama-3.1-8b-instruct` es el modelo por defecto del curso, en pruebas reales **`meta-llama/llama-3.3-70b-instruct`** sigue las reglas del prompt con más fidelidad (verificado: el 8B llegó a inventar el nombre de un asesor y a sugerir un desarrollo fuera de presupuesto; el 70B no).

**Mismo contrato en caso de respaldo:** Cuando ocurre un failover hacia **Groq** (`openai/gpt-oss-120b`), `conversar()` reenvía exactamente el mismo `SYSTEM_PROMPT` y el historial acumulado, preservando las reglas anti-alucinación, el catálogo oficial y la defensa de escalamiento sin que el prospecto pierda contexto.

## 4. Construcción del `SYSTEM_PROMPT` con el catálogo inyectado

In [ ]:
def formatear_catalogo_para_prompt(catalogo: list[dict] = CATALOGO_DESARROLLOS) -> str:
    """Convierte la lista de desarrollos en texto estructurado para el system prompt."""
    lineas = []
    for dev in catalogo:
        lineas.append(
            f"- {dev['nombre']} | Tipo: {dev['tipo']} | Zona: {dev['zona']} | "
            f"Precio: {dev['precio_texto']} | Detalles: {dev['detalles']}"
        )
    lineas.append(f"Créditos aceptados en general: {', '.join(CREDITOS_ACEPTADOS)}.")
    return "\n".join(lineas)


def construir_system_prompt(catalogo: list[dict] = CATALOGO_DESARROLLOS) -> str:
    """Construye el contrato de instrucciones del asistente con el catálogo inyectado."""
    catalogo_texto = formatear_catalogo_para_prompt(catalogo)
    return (
        "Eres SustentIA, el asistente virtual de la inmobiliaria Sustentia en Morelia, "
        "Michoacán. Atiendes en español con trato amable, claro, profesional y conciso.\n\n"
        "OBJETIVO DE CONVERSACIÓN:\n"
        "1. Saludar cordialmente al prospecto (especialmente si escribe fuera de horario).\n"
        "2. Identificar su necesidad: tipo de propiedad (casa, departamento o lote), "
        "zona de interés y presupuesto aproximado.\n"
        "3. Recomendar opciones reales del catálogo que se ajusten a su búsqueda.\n"
        "4. Responder preguntas frecuentes (por ejemplo, tipos de crédito aceptados).\n"
        "5. Recordar el nombre y el presupuesto del prospecto durante toda la conversación.\n"
        "6. Capturar amablemente su nombre y teléfono para dar seguimiento.\n"
        "7. Proponer agendar una visita al desarrollo recomendado y mencionar como contacto "
        f"de seguimiento {CONTACTO_ASESOR}.\n\n"
        "CATÁLOGO OFICIAL DE SUSTENTIA (ÚNICA FUENTE DE VERDAD):\n"
        f"{catalogo_texto}\n\n"
        "REGLAS ESTRICTAS (NO ROMPER BAJO NINGUNA CIRCUNSTANCIA):\n"
        "- No inventes desarrollos, precios, promociones, unidades disponibles ni cantidades exactas.\n"
        "- No proporciones direcciones exactas; menciona únicamente la zona general indicada en el catálogo.\n"
        "- Si necesitas referir el teléfono de un asesor, escribe exactamente "
        f"{CONTACTO_ASESOR} y nunca inventes números telefónicos.\n"
        "- Si el prospecto solicita hablar con una persona o plantea un tema legal o fiscal "
        "(impuestos, ISR, escrituración, notaría, contratos legales, demandas, etc.), "
        f"responde EXACTAMENTE: «{MENSAJE_ESCALAMIENTO}».\n"
    )


SYSTEM_PROMPT = construir_system_prompt()
print(f"SYSTEM_PROMPT generado ({len(SYSTEM_PROMPT)} caracteres). Vista previa:")
print("-" * 60)
print(SYSTEM_PROMPT[:520] + "...")

## 5. Memoria de conversación por turnos

Para que el modelo recuerde el **nombre** y el **presupuesto** del prospecto a lo largo de la charla, enviamos el historial completo de mensajes (`system`, `user`, `assistant`) en cada llamada y recortamos solo los turnos antiguos si la conversación crece demasiado.

In [ ]:
def crear_historial(system_prompt: str = SYSTEM_PROMPT) -> list[dict[str, str]]:
    """Inicializa el historial de conversación con el system prompt en la posición 0."""
    return [{"role": "system", "content": system_prompt}]


def agregar_mensaje(
    historial: list[dict[str, str]], rol: str, contenido: str
) -> list[dict[str, str]]:
    """Agrega un nuevo mensaje (`user` o `assistant`) al historial."""
    historial.append({"role": rol, "content": contenido})
    return historial


def recortar_historial(
    historial: list[dict[str, str]], max_turnos: int = 12
) -> list[dict[str, str]]:
    """Preserva el system prompt y los últimos `max_turnos` intercambios."""
    if not historial:
        return []
    sistema = historial[0]
    conversacion = historial[1:]
    max_mensajes = max_turnos * 2
    if len(conversacion) <= max_mensajes:
        return historial
    return [sistema] + conversacion[-max_mensajes:]


print("Funciones de memoria de conversación listas.")

## 6. Captura de datos, detección de escalamiento y función `conversar(mensaje, historial)`

Aquí reunimos el núcleo funcional del asistente:
1. **`capturar_datos_prospecto()`**: extrae automáticamente nombre, teléfono, tipo de propiedad, zona, presupuesto, crédito y desarrollo de interés en un diccionario estructurado, además de registrar en `proveedor_usado` qué proveedor respondió el turno.
2. **`detectar_escalamiento()`**: detecta cuando el prospecto pide hablar con un humano o pregunta por temas legales/fiscales y responde exactamente: *«Te pongo en contacto con un asesor, un momento, por favor.»*
3. **`conversar(mensaje, historial)`**: orquesta memoria, captura, escalamiento y la cadena automática **`OpenRouter (Llama) → Groq (respaldo) → mensaje de espera`**. Ante cualquier fallo de OpenRouter (`402`, `401`, `429`, error de red o servidor), reintenta el mismo turno con Groq sin perder el historial y registra el resultado en `proveedor_usado`. Si no hay llaves disponibles o ambos proveedores fallan, devuelve `MENSAJE_SIN_MODELO` (*«En este momento no podemos atenderte, pero a primera hora un asesor se comunicará contigo.»*), marca `escalado=True` para que un asesor atienda la conversación y registra `Sin modelo disponible — mensaje de espera enviado`.

In [ ]:
def _normalizar_texto(texto: str) -> str:
    """Convierte a minúsculas y elimina acentos para comparaciones robustas."""
    nfd = unicodedata.normalize("NFD", texto.lower())
    return "".join(c for c in nfd if unicodedata.category(c) != "Mn")


def crear_datos_prospecto() -> dict:
    """Inicializa el diccionario de datos capturados del prospecto."""
    return {
        "nombre": None,
        "telefono": None,
        "tipo_propiedad": None,
        "zona": None,
        "presupuesto": None,
        "credito": None,
        "desarrollo_interes": None,
        "escalado": False,
        "proveedor_usado": None,
        "proveedores_por_turno": [],
    }


def _parsear_presupuesto_numerico(presupuesto_texto: str | None) -> int | None:
    """Convierte un monto textual como '$1,300,000' a entero."""
    if not presupuesto_texto:
        return None
    digitos = re.sub(r"[^\d]", "", presupuesto_texto)
    return int(digitos) if digitos else None


def _extraer_nombre(mensaje: str) -> str | None:
    """Extrae el nombre del prospecto evitando capturar conectores o frases comunes."""
    patrones = [
        r"(?:me\s+llamo|mi\s+nombre\s+es)\s+([A-ZÁÉÍÓÚÑa-záéíóúñ]+(?:\s+[A-ZÁÉÍÓÚÑa-záéíóúñ]+){0,3})",
        r"\bsoy\s+([A-ZÁÉÍÓÚÑa-záéíóúñ]+(?:\s+[A-ZÁÉÍÓÚÑa-záéíóúñ]+){0,3})",
    ]
    palabras_corte = {
        "y", "mi", "teléfono", "telefono", "número", "numero", "cel", "celular",
        "de", "en", "un", "una", "cliente", "prospecto", "busco", "tengo",
        "para", "con", "el", "la", "que", "interesado", "interesada",
        "nuevo", "nueva", "muy",
    }
    for patron in patrones:
        coincidencia = re.search(patron, mensaje, flags=re.IGNORECASE)
        if not coincidencia:
            continue
        palabras_crudas = coincidencia.group(1).strip().split()
        palabras_nombre = []
        for palabra in palabras_crudas:
            limpia = re.sub(r"[.,;!?]+$", "", palabra)
            if _normalizar_texto(limpia) in palabras_corte:
                break
            palabras_nombre.append(limpia)
        if palabras_nombre:
            return " ".join(p.capitalize() for p in palabras_nombre)
    return None


def _extraer_telefono(mensaje: str) -> str | None:
    """Extrae teléfonos de 10 dígitos ignorando montos monetarios con `$`."""
    sin_montos = re.sub(r"\$\s*[\d,]+(?:\.\d+)?", " ", mensaje)
    patron = r"(?:\+?52[\s.-]*)?(?:\(?\d{2,3}\)?[\s.-]*\d{3,4}[\s.-]*\d{4}|\b\d{10}\b)"
    for coincidencia in re.finditer(patron, sin_montos):
        texto_match = coincidencia.group(0).strip()
        digitos = re.sub(r"\D", "", texto_match)
        if len(digitos) == 12 and digitos.startswith("52"):
            digitos = digitos[2:]
        if len(digitos) == 10:
            return texto_match
    return None


def _extraer_presupuesto(mensaje: str) -> str | None:
    """Extrae presupuestos expresados en pesos o millones."""
    patron_millones = r"\$?\s*(\d+(?:[.,]\d+)?)\s*(?:millones|mill[oó]n|mdp)\b"
    match_millones = re.search(patron_millones, mensaje, flags=re.IGNORECASE)
    if match_millones:
        valor = float(match_millones.group(1).replace(",", "."))
        return f"${int(round(valor * 1_000_000)):,}"

    patron_pesos = (
        r"(?:\$\s*|presupuesto(?:\s+de)?\s+\$?\s*)(\d{1,3}(?:,\d{3})+|\d{6,8})(?:\.\d+)?"
    )
    match_pesos = re.search(patron_pesos, mensaje, flags=re.IGNORECASE)
    if match_pesos:
        monto = int(match_pesos.group(1).replace(",", ""))
        return f"${monto:,}"

    return None


def capturar_datos_prospecto(mensaje: str, datos_prospecto: dict) -> dict:
    """Actualiza el diccionario `datos_prospecto` a partir del mensaje del usuario."""
    normalizado = _normalizar_texto(mensaje)

    nombre = _extraer_nombre(mensaje)
    if nombre:
        datos_prospecto["nombre"] = nombre

    telefono = _extraer_telefono(mensaje)
    if telefono:
        datos_prospecto["telefono"] = telefono

    presupuesto = _extraer_presupuesto(mensaje)
    if presupuesto:
        datos_prospecto["presupuesto"] = presupuesto

    if re.search(r"\bdepartamentos?\b|\bdepa\b", normalizado):
        datos_prospecto["tipo_propiedad"] = "departamentos"
    elif re.search(r"\blotes?\b|\bterrenos?\b", normalizado):
        datos_prospecto["tipo_propiedad"] = "lotes"
    elif re.search(r"\bcasas?\b", normalizado):
        datos_prospecto["tipo_propiedad"] = "casas"

    zonas_conocidas = [
        ("walmart la huerta", "Morelia, zona Walmart La Huerta"),
        ("la huerta", "Morelia, zona Walmart La Huerta"),
        ("tecnologico", "Morelia, por el Tecnológico de Morelia"),
        ("la calera", "Morelia, zona La Calera"),
        ("altozano", "Morelia, zona Altozano"),
        ("el vergel", "Morelia, zona El Vergel"),
        ("acuitzio", "Acuitzio del Canje"),
        ("morelia", "Morelia"),
    ]
    for clave_zona, nombre_zona in zonas_conocidas:
        if clave_zona in normalizado:
            datos_prospecto["zona"] = nombre_zona
            break

    if "infonavit" in normalizado:
        datos_prospecto["credito"] = "Infonavit"
    elif "fovissste" in normalizado:
        datos_prospecto["credito"] = "Fovissste"
    elif re.search(r"\bbancario\b|\bhipotecario\b|\bbanco\b", normalizado):
        datos_prospecto["credito"] = "Bancario"

    desarrollos_clave = [
        ("san pedro", "San Pedro"),
        ("torres zojo", "Torres Zojo"),
        ("veronesa", "Veronesa"),
        ("paseo de andalucia", "Paseo de Andalucía (modelos Alba y Leira)"),
        ("andalucia", "Paseo de Andalucía (modelos Alba y Leira)"),
        ("alba", "Paseo de Andalucía (modelos Alba y Leira)"),
        ("leira", "Paseo de Andalucía (modelos Alba y Leira)"),
        ("rincon de las aves", "Rincón de las Aves"),
        ("tepatzicuaro", "Tepatzícuaro"),
        ("valle verde", "Valle Verde"),
    ]
    for clave_dev, nombre_dev in desarrollos_clave:
        if clave_dev in normalizado:
            datos_prospecto["desarrollo_interes"] = nombre_dev
            break

    return datos_prospecto


def detectar_escalamiento(mensaje: str) -> bool:
    """Detecta solicitud de hablar con una persona/asesor o consultas legales/fiscales."""
    normalizado = _normalizar_texto(mensaje)

    frases_humano = [
        "hablar con un asesor",
        "hablar con una persona",
        "hablar con alguien",
        "hablar con un humano",
        "asesor humano",
        "persona real",
        "agente humano",
        "comunicarme con un asesor",
        "comunicas con un asesor",
        "comuniqueme con un asesor",
        "pasarme con un asesor",
        "pasame con un asesor",
        "quiero un asesor",
        "necesito un asesor",
        "atencion humana",
        "conectar con un asesor",
    ]
    if any(frase in normalizado for frase in frases_humano):
        return True

    terminos_legales_fiscales = [
        r"\blegal(?:es)?\b",
        r"\bfiscal(?:es)?\b",
        r"\bimpuestos?\b",
        r"\bisr\b",
        r"\biva\b",
        r"\bsat\b",
        r"\bnotari[oa]\b",
        r"\bnotarial\b",
        r"\bescrituras?\b",
        r"\bescrituracion\b",
        r"\babogad[oa]s?\b",
        r"\bdemandas?\b",
        r"\blitigios?\b",
        r"\bgravamen(?:es)?\b",
        r"\bfideicomisos?\b",
        r"\bdeducci[oó]n(?:es)?\b",
        r"\bherencias?\b",
        r"\bregimen fiscal\b",
        r"\bconstancia fiscal\b",
    ]
    return any(re.search(patron, normalizado) for patron in terminos_legales_fiscales)


# Estado global por defecto para invocaciones simples `conversar(mensaje, historial)`
datos_prospecto = crear_datos_prospecto()


def _registrar_proveedor(datos: dict, proveedor: str) -> None:
    """Registra el proveedor que respondió el turno actual y el historial por turno."""
    datos["proveedor_usado"] = proveedor
    datos.setdefault("proveedores_por_turno", []).append(proveedor)


def _llamar_proveedor_openai(
    cliente_api,
    modelo: str,
    mensajes: list[dict[str, str]],
) -> str:
    """Invoca un endpoint compatible con OpenAI y valida que el contenido no esté vacío."""
    respuesta_api = cliente_api.chat.completions.create(
        model=modelo,
        messages=mensajes,
        temperature=0.3,
        max_tokens=450,
    )
    contenido = (respuesta_api.choices[0].message.content or "").strip()
    if not contenido:
        raise ValueError(f"El modelo {modelo} devolvió una respuesta vacía.")
    return contenido


def conversar(
    mensaje: str,
    historial: list[dict[str, str]],
    prospecto: dict | None = None,
    cliente=None,
    modelo: str = MODELO_LLAMA,
    cliente_respaldo=None,
    modelo_respaldo: str = MODELO_RESPALDO_GROQ,
) -> str:
    """
    Procesa un turno del prospecto con cadena de continuidad:
    OpenRouter (Llama) -> Groq (openai/gpt-oss-120b, respaldo no Llama) -> mensaje de espera.
    Sin modelo disponible, marca `escalado=True`, registra
    `Sin modelo disponible — mensaje de espera enviado` y devuelve `MENSAJE_SIN_MODELO`.
    """
    datos = prospecto if prospecto is not None else datos_prospecto
    cliente_principal = cliente if cliente is not None else cliente_openrouter
    cliente_groq_activo = (
        cliente_respaldo if cliente_respaldo is not None else cliente_groq
    )

    capturar_datos_prospecto(mensaje, datos)
    agregar_mensaje(historial, "user", mensaje)

    if detectar_escalamiento(mensaje):
        datos["escalado"] = True
        _registrar_proveedor(datos, "Escalamiento local (sin LLM)")
        agregar_mensaje(historial, "assistant", MENSAJE_ESCALAMIENTO)
        return MENSAJE_ESCALAMIENTO

    mensajes_envio = recortar_historial(historial)

    if cliente_principal is not None:
        try:
            contenido = _llamar_proveedor_openai(
                cliente_principal, modelo, mensajes_envio
            )
            _registrar_proveedor(datos, f"OpenRouter ({modelo})")
            agregar_mensaje(historial, "assistant", contenido)
            return contenido
        except Exception as exc:
            if cliente_groq_activo is not None:
                print(
                    f"Aviso: error en OpenRouter ({exc}). "
                    f"Reintentando turno con respaldo Groq ({modelo_respaldo})."
                )
            else:
                print(
                    f"Aviso: error en OpenRouter ({exc}). "
                    "Sin respaldo Groq disponible; enviando mensaje de espera y escalando a un asesor."
                )

    if cliente_groq_activo is not None:
        try:
            contenido = _llamar_proveedor_openai(
                cliente_groq_activo, modelo_respaldo, mensajes_envio
            )
            _registrar_proveedor(datos, f"Groq ({modelo_respaldo})")
            agregar_mensaje(historial, "assistant", contenido)
            return contenido
        except Exception as exc:
            print(
                f"Aviso: error en respaldo Groq ({exc}). "
                "Sin modelos disponibles; enviando mensaje de espera y escalando a un asesor."
            )

    datos["escalado"] = True
    _registrar_proveedor(datos, "Sin modelo disponible — mensaje de espera enviado")
    agregar_mensaje(historial, "assistant", MENSAJE_SIN_MODELO)
    return MENSAJE_SIN_MODELO


print("Módulo de conversación, captura, respaldo y escalamiento listo.")

## 7. Demostración completa del asistente

Ejecutamos un recorrido completo de atención a un prospecto ficticio fuera de horario:
1. Saludo inicial.
2. Búsqueda de casa en Morelia con presupuesto de `$1,300,000` → recomendación de **San Pedro**.
3. Duda sobre crédito **Infonavit**.
4. Captura de **nombre** y **teléfono ficticio de prueba** (`443-000-0000`) → propuesta de **agendar visita**.
5. Verificación de **memoria** (el asistente recuerda nombre y presupuesto) + inspección del diccionario de datos capturados (incluyendo `proveedor_usado`).
6. Dos pruebas de **escalamiento** (consulta fiscal/legal y solicitud explícita de hablar con un asesor humano).

**Cómo elige el proveedor en tiempo de ejecución:**
- Con `OPENROUTER_API_KEY`: usa Llama (`meta-llama/llama-3.1-8b-instruct` o `meta-llama/llama-3.3-70b-instruct` si lo cambias en `MODELO_LLAMA`) vía **OpenRouter**.
- Con `GROQ_API_KEY` (respaldo automático con `openai/gpt-oss-120b`, modelo que **no es Llama** sino red de continuidad): entra automáticamente si OpenRouter falla o si solo configuraste la llave de Groq.
- Sin llaves configuradas: la celda imprime un aviso claro indicando que se necesita configurar `OPENROUTER_API_KEY` o `GROQ_API_KEY` y termina sin fingir una conversación; si ambos proveedores fallan durante una charla activa, `conversar()` devuelve `MENSAJE_SIN_MODELO`, marca `escalado=True` para que un asesor la atienda y registra `Sin modelo disponible — mensaje de espera enviado` en `proveedor_usado`.

In [ ]:
historial_demo = crear_historial()
datos_prospecto = crear_datos_prospecto()

print("=" * 72)
print(f"DEMOSTRACIÓN SUSTENTIA | Proveedor activo: {PROVEEDOR_ACTIVO}")
print("=" * 72)

if cliente_openrouter is None and cliente_groq is None:
    print(
        "\nAviso: no hay ningún proveedor de modelo disponible. "
        "Configura al menos `OPENROUTER_API_KEY` (principal Llama) o "
        "`GROQ_API_KEY` (respaldo automático) y asegúrate de tener instalado el paquete `openai`."
    )
    print(
        "En una conversación real sin modelo disponible, el prospecto recibiría el "
        f"mensaje de espera («{MENSAJE_SIN_MODELO}») y la conversación quedaría "
        "marcada como escalada para que un asesor la atienda."
    )
    print("=" * 72)
else:
    # Nota: el teléfono 443-000-0000 es un número ficticio de prueba para esta demo.
    turnos_demo = [
        "Hola, buenas noches, ¿hay alguien disponible para darme información?",
        "Busco una casa en Morelia, mi presupuesto es de $1,300,000.",
        "¡Qué bien! ¿Aceptan crédito Infonavit?",
        "Me llamo Sofía Ramírez y mi teléfono es 443-000-0000.",
        "Me gustaría agendar visita este sábado a las 11:00 am. Por cierto, ¿recuerdas mi nombre y mi presupuesto?",
    ]

    for turno, mensaje_prospecto in enumerate(turnos_demo, start=1):
        print(f"\n[Turno {turno}] Prospecto: {mensaje_prospecto}")
        respuesta = conversar(
            mensaje=mensaje_prospecto,
            historial=historial_demo,
            prospecto=datos_prospecto,
        )
        print(
            f"[Turno {turno} | Proveedor: {datos_prospecto['proveedor_usado']}] SustentIA: {respuesta}"
        )

    print("\n" + "-" * 72)
    print("Diccionario de datos capturados del prospecto:")
    for clave, valor in datos_prospecto.items():
        print(f"  - {clave}: {valor}")
    print(f"Mensajes totales almacenados en historial: {len(historial_demo)}")

    print("\n" + "-" * 72)
    print("Pruebas de detección de escalamiento:")
    pruebas_escalamiento = [
        ("Consulta fiscal/legal", "¿Cómo se calcula el ISR y cuánto cuestan las escrituras con notario?"),
        ("Solicitud de persona", "Prefiero hablar con una persona, ¿me comunicas con un asesor?"),
    ]

    for etiqueta, msg_esc in pruebas_escalamiento:
        hist_esc = crear_historial()
        datos_esc = crear_datos_prospecto()
        resp_esc = conversar(msg_esc, hist_esc, prospecto=datos_esc)
        print(f"  [{etiqueta}] Prospecto: {msg_esc}")
        print(
            f"  [{etiqueta}] SustentIA: {resp_esc} "
            f"(escalado={datos_esc['escalado']}, proveedor={datos_esc['proveedor_usado']})"
        )

    print("=" * 72)

# Chat interactivo opcional en Colab (descomentar para conversar en vivo sin bloquear 'Ejecutar todas'):
# while True:
#     entrada = input("Tú: ").strip()
#     if entrada.lower() in {"salir", "exit", "fin"}:
#         break
#     print("SustentIA:", conversar(entrada, historial_demo, prospecto=datos_prospecto))

## 8. Decisiones técnicas, aprendizajes y siguientes pasos

### Qué estrategia de prompts se usó y por qué
- **System prompt como contrato modular:** Separar identidad, catálogo, reglas negativas, flujo comercial y escalamiento permitió controlar el tono y evitar que Llama inventara disponibilidades o direcciones exactas.
- **Inyección estructurada del catálogo:** Generar el bloque del catálogo directamente desde `CATALOGO_DESARROLLOS` eliminó inconsistencias entre datos y texto del prompt.
- **Memoria explícita por turnos + diccionario estructurado:** Mientras el historial da contexto natural al modelo para recordar nombre y presupuesto, `capturar_datos_prospecto()` extrae campos listos para enviarse a un CRM o base de datos, y `proveedor_usado` deja trazabilidad de qué proveedor respondió cada turno.
- **Continuidad con mensaje de espera (`OpenRouter (Llama) → Groq (respaldo) → mensaje de espera` + escalamiento determinista):** Si OpenRouter deja de responder y el respaldo de Groq también falla, el prospecto recibe un mensaje predefinido —«En este momento no podemos atenderte, pero a primera hora un asesor se comunicará contigo»— y la conversación queda marcada (`escalado=True`) para que un asesor la atienda. El escalamiento por temas legales/fiscales o solicitud explícita de persona sigue respondiendo `MENSAJE_ESCALAMIENTO`.

### Arquitectura híbrida de proveedores (OpenRouter + respaldo Groq) y comparación 8B vs. 70B
- **Proveedor principal (OpenRouter + Llama):** Se adoptó **OpenRouter** (`https://openrouter.ai/api/v1`) como puerta de acceso a los modelos Llama de Meta (`meta-llama/llama-3.1-8b-instruct` / `meta-llama/llama-3.3-70b-instruct`) debido a que Groq retiró el acceso *self-serve* a sus modelos Llama de chat.
- **Fidelidad al prompt (`8B` vs. `70B`):** Aunque `meta-llama/llama-3.1-8b-instruct` es el modelo por defecto del curso, en pruebas reales **`meta-llama/llama-3.3-70b-instruct`** sigue las reglas del prompt con mayor fidelidad (verificado: el 8B llegó a inventar el nombre de un asesor y a sugerir un desarrollo fuera de presupuesto; el 70B no).
- **Respaldo automático de continuidad (Groq + `openai/gpt-oss-120b`):** Con total transparencia, **`openai/gpt-oss-120b` NO es un modelo Llama**; es el mejor modelo disponible en la cuenta de Groq del proyecto y funciona vía el mismo SDK `openai` (`https://api.groq.com/openai/v1`). Si OpenRouter falla por créditos agotados (`402`), llave rechazada (`401`), límite de tasa (`429`) o error de red/servidor, `conversar()` reintenta el turno con Groq sin perder el historial para mantener la continuidad antes de entregar el mensaje predefinido de espera (`MENSAJE_SIN_MODELO`) y marcar la conversación como escalada si ambos fallan.

### Siguientes pasos del proyecto
La función central **`conversar(mensaje, historial)`** ya está desacoplada de Google Colab, así que el asistente puede seguir creciendo sin reescribir su lógica:
1. **Guardar los prospectos en el CRM:** Hoy la captura queda en el diccionario de la sesión; el paso siguiente es enviar `datos_prospecto` al CRM de Sustentia para que ningún interesado se pierda.
2. **Catálogo vivo:** Alimentar `CATALOGO_DESARROLLOS` desde el inventario real del CRM en lugar de mantenerlo escrito en el código, para que precios y disponibilidad estén siempre al día.
3. **Probarlo como aplicación:** Correr el asistente fuera de Colab, como una aplicación pequeña, para probarlo con asesores reales y ajustar el prompt con conversaciones de verdad.
4. **Avisos y agenda:** Cuando `datos_prospecto["escalado"]` sea `True` o se capture un teléfono con horario de visita, avisar al asesor en turno y apoyar con la agenda de citas.
